### Замерим качество Линейной регрессии после обработки данных не просто на отложенной выборке, но и на Кросс-Валидации на 4 фолдах!

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('data/processed_data.csv', index_col='id')

In [2]:
df.head()

,vendor_id,passenger_count,store_and_fwd_flag,distance_km,log_trip_duration
id,,,,,
id2875421,1,930.399753,0,1.500479,6.122493
id2377394,0,930.399753,0,1.807119,6.498282
id3858529,1,930.399753,0,6.392080,7.661527
id3504673,1,930.399753,0,1.487155,6.063785
id2181028,1,930.399753,0,1.189925,6.077642


#### ! Не перемешивайте данные

In [5]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression

selector = KFold(n_splits=4)

### Шаг №5
### Замерьте качество (MSLE, как и раньше) на Кросс-валидации, 
### используя MSE от log_trip_duration и назначенный selector
### Your code is here
print(selector.get_n_splits()) # Функция класса, показывающая на сколько частей поделили наш датасет

X = df.drop('log_trip_duration', axis=1)
Y = df['log_trip_duration']

test_losses = []
train_losses = []
cross_val_error=[]

for train_index, test_index in selector.split(X):

    x_train, x_test = X.values[train_index], X.values[test_index]
    y_train, y_test = Y.values[train_index], Y.values[test_index]

    model = LinearRegression()
    model.fit(x_train, y_train)

    test_losses.append(np.mean((model.predict(x_test) - y_test) ** 2))
    train_losses.append(np.mean((model.predict(x_train) - y_train) ** 2))
    


print(f"MSLE на Кросс-валидации (test): {round(np.mean(test_losses), 3)}")
print(f"MSLE на train: {round(np.mean(train_losses), 3)}")

4
MSLE на Кросс-валидации (test): 0.426
MSLE на train: 0.424


## Поработал один из хитрых гномов!

В отличие от своих собратьев, третий гном оказался тем еще бездельником в школьные годы, но все равно страстно желал во всем догнать первых двух. И сейчас, желая помочь им в построении модели по предсказанию длительности поездки такси, добавил в данные 20 зашифрованных фичей (их смысл нам не рассказали: какая-то секретная информация о водителях).

Гном думал следующим образом: "Ну не может же модель стать хуже! А тут вот авось и мое нововведение уменьшит ошибку в разы! Тогда и меня станут звать на гномий  data-саммит."

Проверим на кросс-валидации, насколько гном оказался прав!

In [10]:
new_data = pd.read_csv('data/new_data.csv', index_col='id')

In [11]:
new_data.head()

,vendor_id,passenger_count,store_and_fwd_flag,distance_km,log_trip_duration,feature_1,feature_2,feature_3,feature_4,feature_5,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20
id,,,,,,,,,,,,,,,,,,,,,
id2875421,1,930.399753,0,1.500479,6.122493,1,1,1,1,1,...,0,0,0,0,0,1.500479,2.251437,3.378234,5.068969,7.605881
id2377394,0,930.399753,0,1.807119,6.498282,0,0,0,0,0,...,0,0,0,0,0,1.807119,3.265681,5.901475,10.664670,19.272331
id3858529,1,930.399753,0,6.392080,7.661527,1,1,1,1,1,...,0,0,0,0,0,6.392080,40.858690,261.172025,1669.432545,10671.146803
id3504673,1,930.399753,0,1.487155,6.063785,1,1,1,1,1,...,0,0,0,0,0,1.487155,2.211629,3.289035,4.891303,7.274125
id2181028,1,930.399753,0,1.189925,6.077642,1,1,1,1,1,...,0,0,0,0,0,1.189925,1.415923,1.684842,2.004837,2.385606


In [12]:
### Шаг №6
### Замерьте качество (MSLE, как и раньше) на Кросс-валидации, 
### используя MSE от log_trip_duration и назначенный ранее selector

X_new = new_data.drop('log_trip_duration', axis=1)
Y_new = new_data['log_trip_duration']

test_losses_new = []
train_losses_new = []

for train_index_new, test_index_new in selector.split(X_new):

    x_train_new, x_test_new = X_new.values[train_index_new], X_new.values[test_index_new]
    y_train_new, y_test_new = Y_new.values[train_index_new], Y_new.values[test_index_new]

    model_new = LinearRegression()
    model_new.fit(x_train_new, y_train_new)

    test_losses_new.append(np.mean((model_new.predict(x_test_new) - y_test_new) ** 2))
    train_losses_new.append(np.mean((model_new.predict(x_train_new) - y_train_new) ** 2))
    


print(f"MSLE на Кросс-валидации (test) 3 гнома: {round(np.mean(test_losses_new), 3)}")
print(f"MSLE на train 3 гнома: {round(np.mean(train_losses_new), 3)}")
#print(f"MSLE на Кросс-валидации: {round(np.mean(cross_val_error), 3)}")

MSLE на Кросс-валидации (test) 3 гнома: 0.633
MSLE на train 3 гнома: 0.632


В линейной алгербре зачастую используют понятие **ранга матрицы**. Оно соответствует кол-ву линейно независимых столбцов в матрице. Иными словами, позволяет оценить, есть ли избыток информации в нашем датафрейме. Если ранг матрицы меньше, чем кол-во используемых столбцов, то некоторые фичи следует удалить, ведь иначе возникает ситуация строгой мультиколлинеарности.

Чтобы замерить ранг в наших матрицах объект-признак, можно воспользоваться функцией numpy.linalg.matrix_rank

Константным признаком в данном упражнении можно пренебречь.

In [13]:
### Создайте переменные rank_processed, rank_new 
### Соответственно равные рангу изначальной матрицы
### с данными и рангу матрицы третьего гнома

### Your code is here
rank_processed = np.linalg.matrix_rank(X)
rank_new = np.linalg.matrix_rank(X_new)




In [14]:
### Создайте переменные num_features_processed, num_features_new
### Соответственно равные кол-ву фичей в изначальной матрицы
### с данными и кол-ву фичей у третьего гнома

### Your code is here

num_features_processed = X.shape[1]
num_features_new = X_new.shape[1]


In [15]:
### Шаг №7
print(f"В первой модели всего фичей: {num_features_processed}, - а ранг равен {rank_processed}")

print(f"Во второй модели всего фичей: {num_features_new}, - а ранг равен {rank_new}")

В первой модели всего фичей: 4, - а ранг равен 4
Во второй модели всего фичей: 24, - а ранг равен 5


Не кажется ли нам, что из-за новых 20 фичей появилась проблема мультиколлинеарности? Как поступить гному, чтобы, с одной стороны, получить адекватное качество, а с другой стороны, не повредить свое самолюбие и не убирать новые признаки?

Верно! Например, с помощью регуляризации.

Найдите такой параметр регуляризации $\alpha$ для Ridge и Lasso случая, чтобы ошибка MSLE на кросс-валидации оказалась строго меньше 0.4

**ALARM**: используйте процедуру масштабирования данных (воспользуйтесь методом MinMaxScaler) перед тем как применить регуляризацию. Важно - чтобы сохранить концепцию независимости обучения на трейне и на тесте, на каждой итерации кросс-валидации необходимо замерять параметры стандартизации исключительно на трейне, а потом применять на валидационном фолде.

In [17]:
### Пример, как это можно сделать в цикле
### То есть обучить Lasso, учитывая масштабирование
### Исключительно на трейне на каждой итерации



X = new_data.drop('log_trip_duration', axis=1)
Y = new_data['log_trip_duration']

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Lasso, Ridge

scores = []

for train_index, test_index in selector.split(X):
    
    X_train, X_test = X.values[train_index], X.values[test_index]
    Y_train, Y_test = Y.values[train_index], Y.values[test_index]
    
    ### Фитим исключительно на трейне!
    scaler = MinMaxScaler()
    scaler.fit(X_train)
    
    ### Применяем обученный scaler на трейн и тест
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    ### max_iter иногда требуется ставить побольше, 
    ### особенно когда данных много и/или они сложные
    ### этот параметр регулирует верхнюю границу кол-ва
    ### итераций во время обучения
    ### подробнее - в документации
    
    ### По дефолту здесь параметр регуляризации alpha=1
    
    model_lasso = Lasso(max_iter=100000) 
    model_lasso.fit(X_train_scaled, Y_train)
    
    predictions = model_lasso.predict(X_test_scaled)
    
    scores.append(np.mean((predictions - Y_test)**2))

    
print(f"MSLE на Кросс-валидации равен: {np.mean(scores)}")

MSLE на Кросс-валидации равен: 0.6332330617999488


In [18]:
### Теперь найдите оптимальный параметр регуляризации
### для случая Ridge
### Напомним: ошибка на кросс-валидации должно быть 
### строго меньше 0.4

### Шаг №8
### Your code is here
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
        ('scaler', MinMaxScaler()),
        ('Ridge', Ridge(max_iter=100000))
    ])

alphas = np.linspace(0.001, 0.05, 20)

param_grid = {"Ridge__alpha": alphas}

search = GridSearchCV(pipe, param_grid, cv=selector, scoring='neg_mean_squared_error')

search.fit(X,Y)

print(f"Лучший параметр (CV score = {search.best_score_:.5f}):")
print(search.best_params_)


Лучший параметр (CV score = -0.38244):
{'Ridge__alpha': 0.01131578947368421}


In [19]:
### Найдите оптимальный параметр регуляризации
### для случая Lasso
### Напомним: ошибка на кросс-валидации должно быть 
### строго меньше 0.4

### Шаг №9
### Your code is here
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
        ('scaler', MinMaxScaler()),
        ('Lasso', Lasso(max_iter=100000))
    ])

alphas = np.linspace(0.00001, 0.0001, 5)

param_grid = {"Lasso__alpha": alphas}


search = GridSearchCV(pipe, param_grid, cv=selector, scoring='neg_mean_squared_error')

search.fit(X,Y)

print(f"Лучший параметр (CV score = {search.best_score_:.5f}):")
print(search.best_params_)

Лучший параметр (CV score = -0.39848):
{'Lasso__alpha': 1e-05}


In [20]:
search.cv_results_

{'mean_fit_time': array([13.79591721,  4.63845158,  2.37333906,  1.09983391,  1.03346223]),
 'std_fit_time': array([9.38335337, 1.5828168 , 0.67782262, 0.03288864, 0.03015247]),
 'mean_score_time': array([0.1051026 , 0.11609346, 0.1054855 , 0.10070246, 0.10014564]),
 'std_score_time': array([0.0125073 , 0.00500609, 0.00704267, 0.00206472, 0.00076017]),
 'param_Lasso__alpha': masked_array(data=[1e-05, 3.2500000000000004e-05, 5.5e-05, 7.75e-05,
                    0.0001],
              mask=[False, False, False, False, False],
        fill_value=1e+20),
 'params': [{'Lasso__alpha': 1e-05},
  {'Lasso__alpha': 3.2500000000000004e-05},
  {'Lasso__alpha': 5.5e-05},
  {'Lasso__alpha': 7.75e-05},
  {'Lasso__alpha': 0.0001}],
 'split0_test_score': array([-0.42743733, -0.41358566, -0.40048097, -0.39328924, -0.39195564]),
 'split1_test_score': array([-0.38753012, -0.39236359, -0.39457458, -0.39709672, -0.40010651]),
 'split2_test_score': array([-0.39542873, -0.40287983, -0.40909585, -0.41633916,